In [ ]:
#| hide
from drona.core import *
from drona.rounds import *

# drona

> Train agents to choose and use the right tools.

An agent imitates the routes its context already shows it. Drona curates that context. It reads the
session archives Ramabana, Claude Code, and Codex leave behind, scores the tool routes it finds
there, and turns one reviewed conversation into the opening history of the next session.

A round moves through three states. `capture` reads a finished session and writes it as an Aidialog
notebook marked `review`. A person edits that notebook in Leela, cutting the detours and anything
private. `accept` records who signed it off. Only then will anything compile: an unaccepted round
reaches no host.

## Install

```sh
pip install drona
```

## Score the routes in an archive

`assess_turn` reads what the agent did. It never infers quality from what the agent said about what
it did, so every finding points at a recorded call.

In [ ]:
turn = {'prompt': 'Use fossick to research https://github.com/AnswerDotAI/llmdojo',
        'activity': [{'tool': 'web_search', 'ok': True, 'args': {'query': 'llmdojo'}}]}
assess_turn(turn)

The prompt named a repository and named the tool that reads one, so reaching for a general web
search first costs the route twenty points. Taking the intended route costs nothing.

In [ ]:
turn['activity'] = [{'tool': 'run_shell', 'ok': True,
                     'args': {'command': 'fossick read-gh-repo https://github.com/AnswerDotAI/llmdojo'}}]
assess_turn(turn)

`drona` scores the whole Ramabana archive as one corpus and prints JSON. `--session` narrows it to
one session, named by id, by any unambiguous prefix, or as `latest`.

```sh
drona
drona --session latest
```

Only turns Ramabana recorded as `complete` are scored. A turn somebody stopped part way is not a
route worth judging, let alone imitating; `--every-state` includes them anyway.

## Curate a round

Capture reads an archive after the conversation has ended, so hold the conversation first and quit.

```sh
ramabana --root /path/to/project
drona-capture training/github-research.ipynb
leela training
```

Open the notebook in Leela and edit it down to the route a later model should imitate. Then accept
it under your own name.

```sh
drona-accept training/github-research.ipynb Karthik
```

Acceptance marks the notebook and writes `github-research.json` beside it as derived canonical
history. Here is that whole cycle against a throwaway archive.

In [ ]:
import json, tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp())
archive = tmp/'agent-history.jsonl'
archive.write_text(json.dumps({
    'session': 'sess-aaa', 'state': 'complete',
    'prompt': 'Use fossick to research the AnswerDotAI llmdojo github repository',
    'reply': 'FOSSICK read the repository directly, so its source files are the evidence.',
    'activity': [{'action_id': 'a0', 'tool': 'run_shell', 'ok': True,
                  'args': {'command': 'fossick read-gh-repo https://github.com/AnswerDotAI/llmdojo'},
                  'detail': '# llmdojo\nLLM coding agents imitate what their context shows.'}]}) + '\n')

round = capture(tmp/'github-research.ipynb', history=archive)
accept(round, 'Karthik')

## Open a chat on a reviewed round

`warm_start` returns ordinary Urai history. Pass it to any Urai or Rishi chat as `messages=`, or
call `prepare_chat` on an empty one.

In [ ]:
[(m['role'], m.get('name')) for m in warm_start(round)]

## Start the next session on it

Ramabana takes no prepared history on its command line, so Drona sends the round as one bootstrap
turn and resumes the session that turn creates.

```sh
drona-start training/github-research.ipynb --root /path/to/project
drona-start training/github-research.ipynb --root /path/to/project --launch
```

Without `--launch` it prints the two commands rather than running them.

In [ ]:
for name, cmd in start_round(round, root='/path/to/project').items(): print(name, cmd[:3])

## Move a round between hosts

The Aidialog notebook is the interchange format, so a session can be captured on one host, reviewed
once, and started on another.

```sh
drona-capture training/round.ipynb
drona-capture training/round.ipynb --host claude --cwd /path/to/project
drona-capture training/round.ipynb --host codex  --cwd /path/to/project
```

The same reviewed notebook compiles back out to any of the three.

```sh
drona-export training/round.ipynb ramabana --output training/round.txt
drona-export training/round.ipynb claude --cwd /path/to/project
drona-export training/round.ipynb codex --output training/round-items.json
```

Claude Code resumes the session id the Claude export prints. The Codex export is not resumable,
because llmsurgery has no public rollout writer; its items are still what you want for inspection
and for datasets.

Every one of these refuses a notebook nobody has accepted.

In [ ]:
from fastcore.test import test_fail
unreviewed = capture(tmp/'unreviewed.ipynb', history=archive)
for host in HOSTS: test_fail(lambda: export_round(unreviewed, host), contains='is not accepted')

## Develop

```sh
uv sync --all-extras --group dev
uv run nbdev-export
uv run nbdev-test
uv run nbdev-readme
uv run nbdev-clean
```